In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install Biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 43.8 MB/s eta 0:00:00


In [3]:
!pip install jsonlines

In [4]:
import numpy as np
import pandas as pd
from Bio import SeqIO
import jsonlines
import re

In [55]:
import os
os.chdir('/content/drive/MyDrive/TGPepGM/raw_datasets/')

In [56]:
!pwd

/content/drive/MyDrive/TGPepGM/raw_datasets


In [57]:
def window(sequence, size=25, max_len=50):
    sequences = []
    for i in range(0, len(sequence), size):
        if i + max_len > len(sequence):
            sequences.append(sequence[i:-1])
            continue
        sequences.append(sequence[i:i+max_len])
    return sequences

In [58]:
def keep_five_to_fifth(sequence):
    vocab = ['A','C','D','E','F','G','H','I','K','L','M','N','P','Q','R','S','T','V','W','Y']
    if type(sequence) is not str:
        return True
    for w in sequence.upper():
        if w not in vocab:
            return True
    if len(sequence) < 5 or len(sequence) > 50:
        return True
    return False

In [59]:
def StringHandler(String):
    String = re.sub("[()]","",String)
    String = re.sub("&&", " ", String)
    String = re.sub("&"," ", String)
    String = re.sub("µM", "", String)
    String = re.sub("µg/ml", "", String)
    String = re.sub("gL", "", String)
    String = re.sub("\\[.*?\\]","",String)
    String = re.sub("\?gl", "", String)
    String = re.sub("," ," ",String)
    String = re.sub(";", "", String)
    String = re.sub("µgl", "", String)
    String = re.sub("卤"," ", String)
    String = re.sub("r'\d{1,2}月\d{1,2}日'", "", String)
    return String


<>:9: SyntaxWarning: invalid escape sequence '\?'
<>:14: SyntaxWarning: invalid escape sequence '\d'
<>:9: SyntaxWarning: invalid escape sequence '\?'
<>:14: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipython-input-330/2425726424.py:9: SyntaxWarning: invalid escape sequence '\?'
  String = re.sub("\?gl", "", String)
/tmp/ipython-input-330/2425726424.py:14: SyntaxWarning: invalid escape sequence '\d'
  String = re.sub("r'\d{1,2}月\d{1,2}日'", "", String)


In [60]:
def remove_nan(sequence):
    vocab = ['A','C','D','E','F','G','H','I','K','L','M','N','P','Q','R','S','T','V','W','Y']
    if type(sequence) is not str:
        return True
    for w in sequence.upper():
        if w not in vocab:
            return True
    if len(sequence) < 5:
        return True
    return False


## input datasets

In [61]:
cancerppd2 = pd.read_csv('./CancerPPD2.csv',sep=',',encoding='utf8')
cancerppd2.head()

,id,pmid,year,seq,name,length,lin_cyc,chiral,chem_mod,cter,nter,cell_line,cancer_type,assay,test_time,tissue
0,7722,NaN,2020,LKKWWKKVKGLLGGLLGKVKKVIK,Seq ID No. 17 from patent ID US202000079827A1,24,Linear,L,NaN,Free,Free,NCI-H460,Lung Cancer,Tetrazolium-based assay,2-h,Lung
1,7721,NaN,2020,LKKWWKKVKGLLGGLLGKVKKVIK,Seq ID No. 17 from patent ID US202000079827A1,24,Linear,L,NaN,Free,Free,HOP-062,Lung Cancer,Tetrazolium-based assay,2-h,Lung
2,7720,NaN,2020,LKKWWKKVKGLLGGLLGKVKSVIK,Seq ID No. 16 from patent ID US202000079827A1,24,Linear,L,NaN,Free,Free,Calu-1,Lung Cancer,Tetrazolium-based assay,2-h,Lung
3,7719,NaN,2020,LKKWWKKVKGLLGGLLGKVKSVIK,Seq ID No. 16 from patent ID US202000079827A1,24,Linear,L,NaN,Free,Free,NCI-H460,Lung Cancer,Tetrazolium-based assay,2-h,Lung
4,7718,NaN,2020,LKKWWKKVKGLLGGLLGKVKSVIK,Seq ID No. 16 from patent ID US202000079827A1,24,Linear,L,NaN,Free,Free,HOP-062,Lung Cancer,Tetrazolium-based assay,2-h,Lung


In [62]:
dracp = pd.read_csv('./dracp.csv',sep=',',encoding='utf8')
dracp.head()

,DCTPep_ID,DRAMP_ID,CancerPPD_ID,DBAASP_ID,Cppsite_ID,Peptide_Name,Sequence,Sequence_Length,UniProt_ID,PubChem_CID,...,Dosage_Form/Route,Company,Marketing_Status,Drug_ID,Approval_year,ClinicalTrials.gov_Identifier,Title,Condition_or_disease,Phase,Purpose
0,DCTPep00001,DRAMP02912,Not available,1485,Not available,SMAP-29,RGLRRLGRKIAHGVKKYGPTVLRIIRIA,28,Not available,16130512,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,DCTPep00002,Not available,Not available,Not available,Not available,CA-MA,KWKLFKKIGIGKFLHSAKKF,20,Not available,Not available,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,DCTPep00003,Not available,Not available,Not available,Not available,CA-MA3,KWKLFKKIGPGKFLHSAKKF,20,Not available,Not available,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,DCTPep00004,Not available,Not available,Not available,Not available,CA-MA1,KWKLFKKIKFLHSAKKF,17,Not available,Not available,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,DCTPep00005,Not available,Not available,Not available,Not available,CA-MA2,KWKLFKKIPKFLHSAKKF,18,Not available,Not available,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [63]:
list(dracp.columns)

['DCTPep_ID',
 'DRAMP_ID',
 'CancerPPD_ID',
 'DBAASP_ID',
 'Cppsite_ID',
 'Peptide_Name',
 'Sequence',
 'Sequence_Length',
 'UniProt_ID',
 'PubChem_CID',
 'Origin',
 'Type',
 'Hemolytic_Activity',
 'Cytotoxicity',
 'Binding_Target',
 'Affinity',
 'Mechanism',
 'Nature',
 '*',
 'Predicted_Structure_ID',
 'PDB_ID',
 'Structure',
 'Classification',
 'Helicity',
 'Linear_Cyclic',
 'Disulfide_Bond',
 'N-terminal_Modification',
 'C-terminal_Modification',
 'Other_Modification',
 'Chiral',
 'Mass',
 'Formula',
 'Absent_amino_acids',
 'Common_amino_acids',
 'pI',
 'Basic_residues',
 'Acidic_residues',
 'Net_charge',
 'Polar_residues',
 'Hydrophobic_residues',
 'Hydrophobicity',
 'Boman_Index',
 'Half_Life',
 'Aliphatic_Index',
 'Extinction_Coefficient_cystines',
 'Absorbance_280nm',
 'Literature',
 'Patent_ID',
 'Patent_Title',
 'Other_Information',
 'Other_Published_ID',
 '__source_file',
 'Cell_Line',
 'Disease',
 'Cancer_Classified',
 'Assay',
 'Activity',
 'Testing_Time',
 'DCTPepD_ID',
 '

In [64]:
list(cancerppd2.columns)

['id',
 'pmid',
 'year',
 'seq',
 'name',
 'length',
 'lin_cyc',
 'chiral',
 'chem_mod',
 'cter',
 'nter',
 'cell_line',
 'cancer_type',
 'assay',
 'test_time',
 'tissue']

In [65]:
dctpep = pd.read_excel('./DCTPep.xlsx')
dctpep.head()

,DCTPep_ID,DRAMP_ID,CancerPPD_ID,DBAASP_ID,Cppsite_ID,Peptide_Name,Sequence,Sequence_Length,UniProt_ID,PubChem_CID,...,Boman_Index,Half_Life,Aliphatic_Index,Extinction_Coefficient_cystines,Absorbance_280nm,Literature,Patent_ID,Patent_Title,Other_Information,Other_Published_ID
0,DCTPep00001,DRAMP02912,Not available,1485,Not available,SMAP-29,RGLRRLGRKIAHGVKKYGPTVLRIIRIA,28,Not available,16130512,...,-6364,Mammalian: 1 hour##Yeast: 2 min##E.coli: 2 min,125.36,1490,55.19,10601638++SMAP-29: a potent antibacterial and ...,Not available,Not available,Not available,Not available
1,DCTPep00002,Not available,Not available,Not available,Not available,CA-MA,KWKLFKKIGIGKFLHSAKKF,20,Not available,Not available,...,-1227,Mammalian: 1.3 hour##Yeast: 3 min##E.coli: 2 min,83.00,5500,289.47,10675500++Effects of the hinge region of cecro...,Not available,Not available,Not available,Not available
2,DCTPep00003,Not available,Not available,Not available,Not available,CA-MA3,KWKLFKKIGPGKFLHSAKKF,20,Not available,Not available,...,-1719,Mammalian: 1.3 hour##Yeast: 3 min##E.coli: 2 min,63.50,5500,289.47,10675500++Effects of the hinge region of cecro...,Not available,Not available,Not available,Not available
3,DCTPep00004,Not available,Not available,Not available,Not available,CA-MA1,KWKLFKKIKFLHSAKKF,17,Not available,Not available,...,-1907,Mammalian: 1.3 hour##Yeast: 3 min##E.coli: 2 min,74.71,5500,343.75,10675500++Effects of the hinge region of cecro...,Not available,Not available,Not available,Not available
4,DCTPep00005,Not available,Not available,Not available,Not available,CA-MA2,KWKLFKKIPKFLHSAKKF,18,Not available,Not available,...,-1907,Mammalian: 1.3 hour##Yeast: 3 min##E.coli: 2 min,70.56,5500,323.53,10675500++Effects of the hinge region of cecro...,Not available,Not available,Not available,Not available


In [66]:
list(dctpep.columns)

['DCTPep_ID',
 'DRAMP_ID',
 'CancerPPD_ID',
 'DBAASP_ID',
 'Cppsite_ID',
 'Peptide_Name',
 'Sequence',
 'Sequence_Length',
 'UniProt_ID',
 'PubChem_CID',
 'Origin',
 'Type',
 'Hemolytic_Activity',
 'Cytotoxicity',
 'Binding_Target',
 'Affinity',
 'Mechanism',
 'Nature',
 '*',
 'Predicted_Structure_ID',
 'PDB_ID',
 'Structure',
 'Classification',
 'Helicity',
 'Linear_Cyclic',
 'Disulfide_Bond',
 'N-terminal_Modification',
 'C-terminal_Modification',
 'Other_Modification',
 'Chiral',
 'Mass',
 'Formula',
 'Absent_amino_acids',
 'Common_amino_acids',
 'pI',
 'Basic_residues',
 'Acidic_residues',
 'Net_charge',
 'Polar_residues',
 'Hydrophobic_residues',
 'Hydrophobicity',
 'Boman_Index',
 'Half_Life',
 'Aliphatic_Index',
 'Extinction_Coefficient_cystines',
 'Absorbance_280nm',
 'Literature',
 'Patent_ID',
 'Patent_Title',
 'Other_Information',
 'Other_Published_ID']

In [67]:
dracp.drop(dracp[dracp['Sequence'].map(keep_five_to_fifth)].index, inplace=True)
cancerppd2.drop(cancerppd2[cancerppd2['seq'].map(keep_five_to_fifth)].index, inplace=True)
dctpep.drop(dctpep[dctpep['Sequence'].map(keep_five_to_fifth)].index, inplace=True)

In [68]:
cancerppd2['seq'] = cancerppd2['seq'].map(str.upper)
dracp['Sequence'] = dracp['Sequence'].map(str.upper)
dctpep['Sequence'] = dctpep['Sequence'].map(str.upper)

## construct descriptions and dataset

In [69]:
total = pd.DataFrame(columns=['Description', 'Sequence'])

In [70]:
des = []
seq = []

In [71]:
for i in range(len(cancerppd2['seq'])):
    descri = ""

    # --- 1. Biological Targets (The "Goal") ---
    if type(cancerppd2.iloc[i]['cancer_type']) is str:
        descri += 'cancer_type:' + StringHandler(cancerppd2.iloc[i]['cancer_type']) + ' '

    if type(cancerppd2.iloc[i]['cell_line']) is str:
        descri += 'cell_line:' + StringHandler(cancerppd2.iloc[i]['cell_line']) + ' '

    if type(cancerppd2.iloc[i]['tissue']) is str:
        descri += 'tissue_source:' + StringHandler(cancerppd2.iloc[i]['tissue']) + ' '

    # --- 2. Activity / Efficacy ---
    # In CancerPPD, 'assay' usually contains the IC50/activity values
    if type(cancerppd2.iloc[i]['assay']) is str:
        descri += 'activity:' + StringHandler(cancerppd2.iloc[i]['assay']) + ' '

    # --- 3. Structural Constraints ---
    if type(cancerppd2.iloc[i]['lin_cyc']) is str:
        descri += 'topology:' + StringHandler(cancerppd2.iloc[i]['lin_cyc']) + ' '

    if type(cancerppd2.iloc[i]['chiral']) is str:
        descri += 'chirality:' + StringHandler(cancerppd2.iloc[i]['chiral']) + ' '

    # Check if length is available (handling both string and number types)
    if str(cancerppd2.iloc[i]['length']) != 'nan':
        descri += 'length:' + StringHandler(str(cancerppd2.iloc[i]['length'])) + ' '

    # --- 4. Modifications ---
    if type(cancerppd2.iloc[i]['nter']) is str:
        descri += 'n_term_mod:' + StringHandler(cancerppd2.iloc[i]['nter']) + ' '

    if type(cancerppd2.iloc[i]['cter']) is str:
        descri += 'c_term_mod:' + StringHandler(cancerppd2.iloc[i]['cter']) + ' '

    if type(cancerppd2.iloc[i]['chem_mod']) is str:
        descri += 'chemical_mod:' + StringHandler(cancerppd2.iloc[i]['chem_mod']) + ' '

    # Append to lists
    des.append(descri)
    seq.append(cancerppd2.iloc[i]['seq'])

In [72]:
cancerppd2_df=pd.DataFrame(columns=['Description', 'Sequence'])

In [73]:
cancerppd2_df['Description']=des
cancerppd2_df['Sequence']=seq

In [74]:
cancerppd2_df.to_csv('/content/drive/MyDrive/TGPepGM/preprocessed_datasets/cancerppd2Preprocessed.csv', index=False)

In [75]:
# Assuming your dataframe is named 'dracp' and you have initialized lists: des = [], seq = []
# Using 'Sequence' as the sequence column based on your first message.

for i in range(len(dracp['Sequence'])):
    descri = ""

    # --- 1. Classification & Origin ---
    if type(dracp.iloc[i]['Classification']) is str:
        descri += 'classification:' + StringHandler(dracp.iloc[i]['Classification']) + ' '

    if type(dracp.iloc[i]['Origin']) is str:
        descri += 'source:' + StringHandler(dracp.iloc[i]['Origin']) + ' '

    # --- 2. Biological Targets (The "Goal") ---
    if type(dracp.iloc[i]['Cancer_Classified']) is str:
        descri += 'cancer_target:' + StringHandler(dracp.iloc[i]['Cancer_Classified']) + ' '

    if type(dracp.iloc[i]['Cell_Line']) is str:
        descri += 'cell_line:' + StringHandler(dracp.iloc[i]['Cell_Line']) + ' '

    if type(dracp.iloc[i]['Binding_Target']) is str:
        descri += 'binding_target:' + StringHandler(dracp.iloc[i]['Binding_Target']) + ' '

    if type(dracp.iloc[i]['Mechanism']) is str:
        descri += 'mechanism:' + StringHandler(dracp.iloc[i]['Mechanism']) + ' '

    # --- 3. Activity & Safety ---
    # Note: Ensure these columns are cast to string if they are numeric in your dataframe
    if str(dracp.iloc[i]['Activity']) != 'nan':
        descri += 'activity:' + StringHandler(str(dracp.iloc[i]['Activity'])) + ' '

    if type(dracp.iloc[i]['Hemolytic_Activity']) is str:
        descri += 'hemolysis:' + StringHandler(dracp.iloc[i]['Hemolytic_Activity']) + ' '

    if type(dracp.iloc[i]['Cytotoxicity']) is str:
        descri += 'cytotoxicity:' + StringHandler(dracp.iloc[i]['Cytotoxicity']) + ' '

    # --- 4. Structural Constraints ---
    if type(dracp.iloc[i]['Linear_Cyclic']) is str:
        descri += 'topology:' + StringHandler(dracp.iloc[i]['Linear_Cyclic']) + ' '

    if type(dracp.iloc[i]['Chiral']) is str:
        descri += 'chirality:' + StringHandler(dracp.iloc[i]['Chiral']) + ' '

    if type(dracp.iloc[i]['Helicity']) is str:
        descri += 'helicity:' + StringHandler(dracp.iloc[i]['Helicity']) + ' '

    if type(dracp.iloc[i]['Disulfide_Bond']) is str:
        descri += 'disulfide_bond:' + StringHandler(dracp.iloc[i]['Disulfide_Bond']) + ' '

    # Handling numeric constraints (Length & Mass) by casting to string safe-check
    if str(dracp.iloc[i]['Sequence_Length']) != 'nan':
        descri += 'length:' + StringHandler(str(dracp.iloc[i]['Sequence_Length'])) + ' '

    if str(dracp.iloc[i]['Mass']) != 'nan':
        descri += 'mass:' + StringHandler(str(dracp.iloc[i]['Mass'])) + ' '

    # --- 5. Modifications ---
    if type(dracp.iloc[i]['N-terminal_Modification']) is str:
        descri += 'n_term_mod:' + StringHandler(dracp.iloc[i]['N-terminal_Modification']) + ' '

    if type(dracp.iloc[i]['C-terminal_Modification']) is str:
        descri += 'c_term_mod:' + StringHandler(dracp.iloc[i]['C-terminal_Modification']) + ' '

    # --- 6. Physicochemical Properties ---
    if str(dracp.iloc[i]['Net_charge']) != 'nan':
        descri += 'charge:' + StringHandler(str(dracp.iloc[i]['Net_charge'])) + ' '

    if str(dracp.iloc[i]['Hydrophobicity']) != 'nan':
        descri += 'hydrophobicity:' + StringHandler(str(dracp.iloc[i]['Hydrophobicity'])) + ' '

    # Append to lists
    des.append(descri)
    seq.append(dracp.iloc[i]['Sequence'])

In [76]:
dracp_df=pd.DataFrame(columns=['Description', 'Sequence'])

In [77]:
dracp_df['Description']=des
dracp_df['Sequence']=seq

In [78]:
dracp_df.to_csv('/content/drive/MyDrive/TGPepGM/preprocessed_datasets/dracpPreprocessed.csv', index=False)

In [79]:
# Assuming your dataframe is named 'dctpep' (or similar)
# and lists des=[], seq=[] are initialized

for i in range(len(dctpep['Sequence'])):
    descri = ""

    # --- 1. Classification & Origin ---
    if type(dctpep.iloc[i]['Classification']) is str:
        descri += 'classification:' + StringHandler(dctpep.iloc[i]['Classification']) + ' '

    if type(dctpep.iloc[i]['Type']) is str:
        descri += 'type:' + StringHandler(dctpep.iloc[i]['Type']) + ' '

    if type(dctpep.iloc[i]['Origin']) is str:
        descri += 'source:' + StringHandler(dctpep.iloc[i]['Origin']) + ' '

    if type(dctpep.iloc[i]['Nature']) is str:
        descri += 'nature:' + StringHandler(dctpep.iloc[i]['Nature']) + ' '

    # --- 2. Biological Targets & Efficacy ---
    if type(dctpep.iloc[i]['Binding_Target']) is str:
        descri += 'binding_target:' + StringHandler(dctpep.iloc[i]['Binding_Target']) + ' '

    if type(dctpep.iloc[i]['Mechanism']) is str:
        descri += 'mechanism:' + StringHandler(dctpep.iloc[i]['Mechanism']) + ' '

    if type(dctpep.iloc[i]['Affinity']) is str:
        descri += 'affinity:' + StringHandler(dctpep.iloc[i]['Affinity']) + ' '

    # --- 3. Safety Profile ---
    if type(dctpep.iloc[i]['Hemolytic_Activity']) is str:
        descri += 'hemolysis:' + StringHandler(dctpep.iloc[i]['Hemolytic_Activity']) + ' '

    if type(dctpep.iloc[i]['Cytotoxicity']) is str:
        descri += 'cytotoxicity:' + StringHandler(dctpep.iloc[i]['Cytotoxicity']) + ' '

    # --- 4. Structural Constraints ---
    if type(dctpep.iloc[i]['Linear_Cyclic']) is str:
        descri += 'topology:' + StringHandler(dctpep.iloc[i]['Linear_Cyclic']) + ' '

    if type(dctpep.iloc[i]['Chiral']) is str:
        descri += 'chirality:' + StringHandler(dctpep.iloc[i]['Chiral']) + ' '

    if type(dctpep.iloc[i]['Structure']) is str:
        descri += 'structure_type:' + StringHandler(dctpep.iloc[i]['Structure']) + ' '

    if type(dctpep.iloc[i]['Helicity']) is str:
        descri += 'helicity:' + StringHandler(dctpep.iloc[i]['Helicity']) + ' '

    if type(dctpep.iloc[i]['Disulfide_Bond']) is str:
        descri += 'disulfide_bond:' + StringHandler(dctpep.iloc[i]['Disulfide_Bond']) + ' '

    # Handling numeric constraints (Length & Mass)
    if str(dctpep.iloc[i]['Sequence_Length']) != 'nan':
        descri += 'length:' + StringHandler(str(dctpep.iloc[i]['Sequence_Length'])) + ' '

    if str(dctpep.iloc[i]['Mass']) != 'nan':
        descri += 'mass:' + StringHandler(str(dctpep.iloc[i]['Mass'])) + ' '

    # --- 5. Modifications ---
    if type(dctpep.iloc[i]['N-terminal_Modification']) is str:
        descri += 'n_term_mod:' + StringHandler(dctpep.iloc[i]['N-terminal_Modification']) + ' '

    if type(dctpep.iloc[i]['C-terminal_Modification']) is str:
        descri += 'c_term_mod:' + StringHandler(dctpep.iloc[i]['C-terminal_Modification']) + ' '

    if type(dctpep.iloc[i]['Other_Modification']) is str:
        descri += 'other_mod:' + StringHandler(dctpep.iloc[i]['Other_Modification']) + ' '

    # --- 6. Physicochemical Properties ---
    # These are numeric in DCTPep, converting to string
    if str(dctpep.iloc[i]['Net_charge']) != 'nan':
        descri += 'charge:' + StringHandler(str(dctpep.iloc[i]['Net_charge'])) + ' '

    if str(dctpep.iloc[i]['Hydrophobicity']) != 'nan':
        descri += 'hydrophobicity:' + StringHandler(str(dctpep.iloc[i]['Hydrophobicity'])) + ' '

    if str(dctpep.iloc[i]['pI']) != 'nan':
        descri += 'pI:' + StringHandler(str(dctpep.iloc[i]['pI'])) + ' '

    if str(dctpep.iloc[i]['Boman_Index']) != 'nan':
        descri += 'boman_index:' + StringHandler(str(dctpep.iloc[i]['Boman_Index'])) + ' '

    if str(dctpep.iloc[i]['Half_Life']) != 'nan':
        descri += 'half_life:' + StringHandler(str(dctpep.iloc[i]['Half_Life'])) + ' '

    if str(dctpep.iloc[i]['Aliphatic_Index']) != 'nan':
        descri += 'aliphatic_index:' + StringHandler(str(dctpep.iloc[i]['Aliphatic_Index'])) + ' '

    # Append to lists
    des.append(descri.strip())
    seq.append(dctpep.iloc[i]['Sequence'])

In [80]:
dctpep_df=pd.DataFrame(columns=['Description', 'Sequence'])

In [81]:
dctpep_df['Description']=des
dctpep_df['Sequence']=seq

In [82]:
dctpep_df.to_csv('/content/drive/MyDrive/TGPepGM/preprocessed_datasets/dctpepPreprocessed.csv', index=False)

In [83]:
total['Description'] = des
total['Sequence'] = seq

In [84]:
len(total)

14086

In [85]:
total.drop(total[total['Sequence'].map(remove_nan)].index,inplace=True)

In [86]:
len(total)

14086

In [88]:
total.to_csv('/content/drive/MyDrive/TGPepGM/final_datasets/total_acp.csv', index=False)

# Construct non-ACP data

In [ ]:
dramp = pd.read_excel('./Antimicrobial_amps.xlsx')
dramp.head()

,DRAMP_ID,Sequence,Sequence_Length,Name,Swiss_Prot_Entry,Family,Gene,Source,Activity,Protein_existence,...,N-terminal_Modification,C-terminal_Modification,Other_Modifications,Stereochemistry,Cytotoxicity,Binding_Traget,Pubmed_ID,Reference,Author,Title
0,DRAMP00005,SLGPAIKATRQVCPKATRFVTVSCKKSDCQ,30,Epicidin 280 (Bacteriocin),O54220,Belongs to the lantibiotic family (Class I bac...,eciA,Staphylococcus epidermidis BN 280 (Gram-positi...,"Antimicrobial, Antibacterial, Anti-Gram+",Protein level,...,Oxypropionylation,Not metioned clearly,There are possible lanthionine(Lan)/3-methylla...,L,No cytotoxicity information found,Not found,9726851,Appl Environ Microbiol. 1998 Sep;64(9):3140-3146.,"Heidrich C, Pag U, Josten M, Metzger J, Jack R...","Isolation, characterization, and heterologous ..."
1,DRAMP00017,VTSWSLCTPGCTSPGGGSNCSFCC,24,Microbisporicin A1 (Bacteriocin),No entry found,Belongs to the lantibiotic family (Class I bac...,Not found,Microbispora corallina (Gram-positive bacteria),"Antimicrobial, Antibacterial",Protein level,...,Free,Amidation and Cyclization,①There are five thioether intramolecular bridg...,L,No cytotoxicity information found,Not found,18215770,Chem Biol. 2008 Jan;15(1):22-31.,"Castiglione F, Lazzarini A, Carrano L, Corti E...",Determining the structure and mode of action o...
2,DRAMP00032,GNGVLKTISHECNMNTWQFLFTCC,24,Ruminococcin A (RumA; Bacteriocin),"P83674, P83676, P83677, Q8VLK0, Q9K381, P83675...",Belongs to the type A lantibiotic family (Clas...,rumA1 AND rumA2 AND,Ruminococcus gnavus & Ruminococcus hansenii (G...,"Antimicrobial, Antibacterial, Anti-Gram+",Protein level,...,Free,Cyclization (possibly),① It is highly probable that (i) Ser9 and the ...,L,No cytotoxicity information found,Cell membrane,11526013##11741840##12089024,Appl Environ Microbiol. 2001 Sep;67(9):4111-41...,"Dabard J, Bridonneau C, Phillipe C, Anglade P,...","Ruminococcin A, a new lantibiotic produced by ..."
3,DRAMP00063,SSSGWLCTLTIECGTIICACR,21,Lantibiotic michiganin-A (Bacteriocin),Q09T02,Belongs to the type B lantibiotic family (Clas...,micA,Clavibacter michiganensis subsp. Michiganensis...,"Antimicrobial, Antibacterial",Protein level,...,Free,Free,All of the threonine residues undergo dehydrat...,L,No cytotoxicity information found,Not found,16957199,Appl Environ Microbiol. 2006 Sep;72(9):5814-5821.,"Holtsmark I, Mantzilas D, Eijsink VG, Brurberg...","Purification, characterization, and gene seque..."
4,DRAMP00068,MSWLNFLKYIAKYGKKAVSAAWKYKGKVLEWLNVGPTLEWVWQKLK...,51,Aureocin A53 (Bacteriocin),Q8GPI4,Belongs to the class II bacteriocin,aucA,Staphylococcus aureus A53 (Gram-positive bacte...,"Antimicrobial, Antibacterial, Anti-Gram+",Protein level,...,Formylation,Free,NaN,L,No cytotoxicity information found,Cell membrane,12054867,J Mol Biol. 2002 Jun 7;319(3):745-756.,"Netz DJ, Pohl R, Beck-Sickinger AG, Selmer T, ...",Biochemical characterisation and genetic analy...


In [ ]:
non_acp_amp = dramp[dramp['Activity']=='Antimicrobial'].copy()

In [ ]:
len(non_acp_amp)

564

In [ ]:
list(non_acp_amp.columns)

['DRAMP_ID',
 'Sequence',
 'Sequence_Length',
 'Name',
 'Swiss_Prot_Entry',
 'Family',
 'Gene',
 'Source',
 'Activity',
 'Protein_existence',
 'Structure',
 'Structure_Description',
 'PDB_ID',
 'Comments',
 'Target_Organism',
 'Hemolytic_activity',
 'Linear/Cyclic/Branched',
 'N-terminal_Modification',
 'C-terminal_Modification',
 'Other_Modifications',
 'Stereochemistry',
 'Cytotoxicity',
 'Binding_Traget',
 'Pubmed_ID',
 'Reference',
 'Author',
 'Title']

In [ ]:
non_acp_amp.drop(non_acp_amp[non_acp_amp['Sequence'].map(keep_five_to_fifth)].index, inplace=True)

In [ ]:
len(non_acp_amp)

346

In [ ]:
non_acp_amp['Sequence'] = non_acp_amp['Sequence'].map(str.upper)

In [ ]:
# Assuming your dataframe is named 'non_acp_amp' (or similar)
# and lists des=[], seq=[] are initialized

for i in range(len(non_acp_amp['Sequence'])):
    descri = ""

    # --- 1. Biological Targets & Function ---
    if type(non_acp_amp.iloc[i]['Activity']) is str:
        descri += 'activity:' + StringHandler(non_acp_amp.iloc[i]['Activity']) + ' '

    if type(non_acp_amp.iloc[i]['Target_Organism']) is str:
        descri += 'target_organism:' + StringHandler(non_acp_amp.iloc[i]['Target_Organism']) + ' '

    if type(non_acp_amp.iloc[i]['Binding_Traget']) is str:  # Note: Keeping the typo from your list
        descri += 'binding_target:' + StringHandler(non_acp_amp.iloc[i]['Binding_Traget']) + ' '

    if type(non_acp_amp.iloc[i]['Family']) is str:
        descri += 'family:' + StringHandler(non_acp_amp.iloc[i]['Family']) + ' '

    # --- 2. Safety Profile ---
    if type(non_acp_amp.iloc[i]['Hemolytic_activity']) is str:
        descri += 'hemolysis:' + StringHandler(non_acp_amp.iloc[i]['Hemolytic_activity']) + ' '

    if type(non_acp_amp.iloc[i]['Cytotoxicity']) is str:
        descri += 'cytotoxicity:' + StringHandler(non_acp_amp.iloc[i]['Cytotoxicity']) + ' '

    # --- 3. Structural Constraints ---
    if type(non_acp_amp.iloc[i]['Structure']) is str:
        descri += 'structure_type:' + StringHandler(non_acp_amp.iloc[i]['Structure']) + ' '

    if type(non_acp_amp.iloc[i]['Linear/Cyclic/Branched']) is str:
        descri += 'topology:' + StringHandler(non_acp_amp.iloc[i]['Linear/Cyclic/Branched']) + ' '

    if type(non_acp_amp.iloc[i]['Stereochemistry']) is str:
        descri += 'chirality:' + StringHandler(non_acp_amp.iloc[i]['Stereochemistry']) + ' '

    # Handling numeric constraints
    if str(non_acp_amp.iloc[i]['Sequence_Length']) != 'nan':
        descri += 'length:' + StringHandler(str(non_acp_amp.iloc[i]['Sequence_Length'])) + ' '

    # --- 4. Modifications ---
    if type(non_acp_amp.iloc[i]['N-terminal_Modification']) is str:
        descri += 'n_term_mod:' + StringHandler(non_acp_amp.iloc[i]['N-terminal_Modification']) + ' '

    if type(non_acp_amp.iloc[i]['C-terminal_Modification']) is str:
        descri += 'c_term_mod:' + StringHandler(non_acp_amp.iloc[i]['C-terminal_Modification']) + ' '

    if type(non_acp_amp.iloc[i]['Other_Modifications']) is str:
        descri += 'other_mod:' + StringHandler(non_acp_amp.iloc[i]['Other_Modifications']) + ' '

    # --- 5. Origin ---
    if type(non_acp_amp.iloc[i]['Source']) is str:
        descri += 'source:' + StringHandler(non_acp_amp.iloc[i]['Source']) + ' '

    # Append to lists
    des.append(descri.strip())
    seq.append(dramp.iloc[i]['Sequence'])

In [ ]:
non_acp = pd.DataFrame({'Description': des, 'Sequence': seq})

In [ ]:
len(non_acp)

346

In [ ]:
non_acp.drop(non_acp[non_acp['Sequence'].map(remove_nan)].index,inplace=True)

In [ ]:
len(non_acp)

345

In [ ]:
non_acp.to_csv('./non_acp_data.csv', index=False)

# Split to train and test

In [89]:
from sklearn.model_selection import train_test_split

In [90]:
total

,Description,Sequence
0,cancer_type:Lung Cancer cell_line:NCI-H460 tis...,LKKWWKKVKGLLGGLLGKVKKVIK
1,cancer_type:Lung Cancer cell_line:HOP-062 tiss...,LKKWWKKVKGLLGGLLGKVKKVIK
2,cancer_type:Lung Cancer cell_line:Calu-1 tissu...,LKKWWKKVKGLLGGLLGKVKSVIK
3,cancer_type:Lung Cancer cell_line:NCI-H460 tis...,LKKWWKKVKGLLGGLLGKVKSVIK
4,cancer_type:Lung Cancer cell_line:HOP-062 tiss...,LKKWWKKVKGLLGGLLGKVKSVIK
...,...,...
14081,classification:Tumor active peptide type:Synth...,FHAVPQSFYT
14082,classification:Tumor active peptide type:Synth...,FHAVPQSFYT
14083,classification:Tumor active peptide type:Synth...,FHAVPQSFYT
14084,classification:Tumor active peptide type:Synth...,KLWCKSSQVPQSR


In [ ]:
train_acp, test_acp = train_test_split(total, test_size=0.2, random_state=42)

In [92]:
print(f"Training set size: {len(train_acp)}")
print(f"Test set size: {len(test_acp)}")

Training set size: 11268
Test set size: 2818


In [93]:
train_acp.to_csv('/content/drive/MyDrive/TGPepGM/final_datasets/train_acp.csv', index=False)
test_acp.to_csv('/content/drive/MyDrive/TGPepGM/final_datasets/test_acp.csv', index=False)